# Session 4 — Alignment: DPO & GRPO

**Duration:** 75 min  
**Repository:** [MiniMind-Colab](https://github.com/your-org/minimind-colab)

| Part | Topic | Time |
|------|-------|------|
| A | Direct Preference Optimization (DPO) | 35 min |
| B | Group Relative Policy Optimization (GRPO) | 35 min |
| — | Wrap-up & Q\&A | 5 min |

In [ ]:
# Cell — Environment setup
!pip install torch transformers datasets --quiet

import math, json, os, time, random, re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
from transformers.activations import ACT2FN
from transformers import PreTrainedModel, GenerationMixin, PretrainedConfig, AutoTokenizer
from transformers.modeling_outputs import MoeCausalLMOutputWithPast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
print(f'Device: {device}')

## Part A — Direct Preference Optimization (35 min)

DPO aligns a language model with human preferences **without** training a separate reward model.

Given pairs of (chosen, rejected) responses to the same prompt, DPO directly optimizes the policy so that
the **chosen** response becomes more likely relative to the **rejected** one — compared to a frozen reference model.

### Key equation

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \left[\log \frac{\pi_\theta(y_w|x)}{\pi_\theta(y_l|x)} - \log \frac{\pi_{\text{ref}}(y_w|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

Where:
- $\pi_\theta$ is the **policy** model (being trained)
- $\pi_{\text{ref}}$ is the **reference** model (frozen copy)
- $y_w$ is the chosen (preferred) response
- $y_l$ is the rejected response
- $\beta$ controls the strength of the preference signal

### Prerequisites from Sessions 1–3

We re-define the model architecture from previous sessions.
These are provided — no changes needed.

In [ ]:
# Cell — Prerequisites: Model architecture (from Sessions 1–3)

class RMSNorm(torch.nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

def precompute_freqs_cis(dim, end=32*1024, rope_base=1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[:xq_.shape[1]].unsqueeze(0).unsqueeze(2)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

    def forward(self, x, freqs_cis, attention_mask=None, past_kv=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim)
        q, k = apply_rotary_emb(q, k, freqs_cis)
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=1)
            v = torch.cat([pv, v], dim=1)
        new_kv = (k, v) if use_cache else None
        if self.num_kv_groups > 1:
            k = k.unsqueeze(3).expand(-1, -1, -1, self.num_kv_groups, -1).reshape(B, -1, self.num_heads, self.head_dim)
            v = v.unsqueeze(3).expand(-1, -1, -1, self.num_kv_groups, -1).reshape(B, -1, self.num_heads, self.head_dim)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=attention_mask, is_causal=(attention_mask is None and T > 1))
        out = attn.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out), new_kv

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden = int(config.hidden_size * 8 / 3)
        hidden = ((hidden + 63) // 64) * 64
        self.gate_proj = nn.Linear(config.hidden_size, hidden, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, config.hidden_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_act]
    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.attention = Attention(config)
        self.feed_forward = FeedForward(config)
        self.attention_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.ffn_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
    def forward(self, x, freqs_cis, attention_mask=None, past_kv=None, use_cache=False):
        h, new_kv = self.attention(self.attention_norm(x), freqs_cis, attention_mask, past_kv, use_cache)
        h = x + h
        out = h + self.feed_forward(self.ffn_norm(h))
        return out, new_kv

class MiniMindConfig(PretrainedConfig):
    model_type = 'minimind'
    def __init__(self, hidden_size=128, num_hidden_layers=2, num_attention_heads=4,
                 num_key_value_heads=2, head_dim=32, vocab_size=32000, max_seq_len=32768,
                 rms_norm_eps=1e-6, rope_theta=1e6, hidden_act='silu', use_moe=False, **kw):
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = head_dim or hidden_size // num_attention_heads
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.use_moe = use_moe
        super().__init__(**kw)

class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    config_class = MiniMindConfig
    def __init__(self, config):
        super().__init__(config)
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([TransformerBlock(i, config) for i in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.output = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.register_buffer('freqs_cis',
            precompute_freqs_cis(config.head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.aux_loss = torch.tensor(0.0)
        self.OUT = MoeCausalLMOutputWithPast

    def forward(self, input_ids=None, labels=None, attention_mask=None,
                past_key_values=None, use_cache=False, **kw):
        h = self.tok_embeddings(input_ids)
        new_kvs = []
        for i, layer in enumerate(self.layers):
            pk = past_key_values[i] if past_key_values else None
            h, nk = layer(h, self.freqs_cis, attention_mask, pk, use_cache)
            new_kvs.append(nk)
        logits = self.output(self.norm(h))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1), ignore_index=-100
            )
        return self.OUT(loss=loss, logits=logits, aux_loss=self.aux_loss,
                        past_key_values=new_kvs if use_cache else None)

print('Model architecture defined ✓')

### 1. Log-Probability Extraction

Before computing the DPO loss we need a helper that converts raw logits
into **per-token log-probabilities** for the tokens that actually appeared
in the sequence.

This is a two-step process:
1. Apply `log_softmax` over the vocabulary dimension to get a valid log-probability distribution.
2. Use `gather` to pick out the log-prob corresponding to each ground-truth token.

In [ ]:
# Cell — logits_to_log_probs implementation

def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)
    log_probs_per_token = torch.gather(log_probs, dim=2, index=labels.unsqueeze(2)).squeeze(-1)
    return log_probs_per_token

print('logits_to_log_probs defined ✓')

### 2. DPO Loss Function

The DPO loss encourages the policy model to assign **higher relative probability**
to chosen responses vs. rejected ones — calibrated against a frozen reference model.

Steps:
1. Mask and sum log-probs per sequence to get sequence-level log-probabilities.
2. Split the batch: first half = chosen, second half = rejected.
3. Compute the policy and reference log-ratios.
4. Apply `logsigmoid` to get the final loss.

In [ ]:
# Cell — dpo_loss implementation

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta):
    ref_log_probs = (ref_log_probs * mask).sum(dim=1)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)

    batch_size = ref_log_probs.shape[0]
    chosen_ref = ref_log_probs[:batch_size // 2]
    reject_ref = ref_log_probs[batch_size // 2:]
    chosen_policy = policy_log_probs[:batch_size // 2]
    reject_policy = policy_log_probs[batch_size // 2:]

    pi_logratios = chosen_policy - reject_policy
    ref_logratios = chosen_ref - reject_ref
    logits = pi_logratios - ref_logratios
    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

print('dpo_loss defined ✓')

### 3. DPO Dataset — Loss Mask Generation

The `DPODataset` loads preference pairs and tokenizes them. A critical piece is the
`generate_loss_mask` method, which marks **only assistant-turn tokens** with 1.

This ensures the DPO loss only considers the assistant's actual response tokens,
not the system prompt or user messages.

The method scans for BOS-assistant markers (`self.bos_id`) and EOS markers
(`self.eos_id`) to identify assistant turn boundaries.

In [ ]:
# Cell — DPODataset implementation

class DPODataset(Dataset):
    def __init__(self, tokenizer, max_length=256, num_samples=100):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        self.bos_id = tokenizer.encode('<bos>assistant\n', add_special_tokens=False)
        if not self.bos_id:
            self.bos_id = [tokenizer.bos_token_id or 1]
        self.eos_id = tokenizer.encode('<eos>\n', add_special_tokens=False)
        if not self.eos_id:
            self.eos_id = [tokenizer.eos_token_id or 2]
        # Generate synthetic preference pairs
        self.samples = []
        for i in range(num_samples):
            prompt_tokens = [random.randint(3, 99) for _ in range(20)]
            chosen_tokens = self.bos_id + [random.randint(3, 99) for _ in range(30)] + self.eos_id
            rejected_tokens = self.bos_id + [random.randint(3, 99) for _ in range(30)] + self.eos_id
            self.samples.append({
                'chosen': prompt_tokens + chosen_tokens,
                'rejected': prompt_tokens + rejected_tokens
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        chosen_ids = sample['chosen'][:self.max_length]
        rejected_ids = sample['rejected'][:self.max_length]
        # Pad to max_length
        chosen_ids = chosen_ids + [self.padding] * (self.max_length - len(chosen_ids))
        rejected_ids = rejected_ids + [self.padding] * (self.max_length - len(rejected_ids))
        chosen_mask = self.generate_loss_mask(chosen_ids)
        rejected_mask = self.generate_loss_mask(rejected_ids)
        x_chosen = torch.tensor(chosen_ids[:-1], dtype=torch.long)
        y_chosen = torch.tensor(chosen_ids[1:], dtype=torch.long)
        mask_chosen = torch.tensor(chosen_mask[1:], dtype=torch.long)
        x_rejected = torch.tensor(rejected_ids[:-1], dtype=torch.long)
        y_rejected = torch.tensor(rejected_ids[1:], dtype=torch.long)
        mask_rejected = torch.tensor(rejected_mask[1:], dtype=torch.long)
        return {
            'x_chosen': x_chosen, 'y_chosen': y_chosen, 'mask_chosen': mask_chosen,
            'x_rejected': x_rejected, 'y_rejected': y_rejected, 'mask_rejected': mask_rejected
        }

    def generate_loss_mask(self, input_ids):
        loss_mask = [0] * len(input_ids)
        i = 0
        while i < len(input_ids):
            if input_ids[i:i + len(self.bos_id)] == self.bos_id:
                start = i + len(self.bos_id)
                end = start
                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_id)] == self.eos_id:
                        break
                    end += 1
                for j in range(start, min(end + len(self.eos_id), self.max_length)):
                    loss_mask[j] = 1
                i = end + len(self.eos_id) if end < len(input_ids) else len(input_ids)
            else:
                i += 1
        return loss_mask

print('DPODataset defined ✓')

### ✅ Milestone 1: DPO Loss Verification

We verify that the DPO loss function produces a valid positive scalar.

In [ ]:
# Cell — ✅ Milestone 1: DPO loss verification

# Setup tokenizer
tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/cosmo2-tokenizer', trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Create a tiny model
config = MiniMindConfig(
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=2,
    num_key_value_heads=2,
    head_dim=32,
    vocab_size=tokenizer.vocab_size,
)
torch.manual_seed(42)
test_model = MiniMindForCausalLM(config).to(device)

# Create synthetic data
B, T, V = 4, 32, tokenizer.vocab_size
dummy_input = torch.randint(0, V, (B, T)).to(device)
dummy_labels = torch.randint(0, V, (B, T)).to(device)
dummy_mask = torch.ones(B, T).to(device)

# Forward pass
with torch.no_grad():
    out = test_model(dummy_input)
    logits = out.logits

# Test logits_to_log_probs
log_probs = logits_to_log_probs(logits, dummy_labels)
assert log_probs.shape == (B, T), f'Shape mismatch: {log_probs.shape} vs ({B}, {T})'
assert (log_probs <= 0).all(), 'Log-probs should be non-positive'
print(f'logits_to_log_probs output shape: {log_probs.shape} ✓')

# Test dpo_loss
ref_lp = logits_to_log_probs(logits, dummy_labels)
# Slightly perturb for policy
policy_logits = logits + 0.1 * torch.randn_like(logits)
policy_lp = logits_to_log_probs(policy_logits, dummy_labels)

loss_val = dpo_loss(ref_lp, policy_lp, dummy_mask, beta=0.1)
print(f'DPO loss value: {loss_val.item():.4f}')

assert not torch.isnan(loss_val), 'DPO loss is NaN!'
assert loss_val.item() > 0, f'DPO loss should be positive, got {loss_val.item()}'
assert loss_val.item() < 10, f'DPO loss too large: {loss_val.item()}'
print(f'✅ Milestone 1 passed — DPO loss is a valid positive scalar: {loss_val.item():.4f}')

In [ ]:
# Cell — Learning rate schedule (provided)

def get_lr(current_step, total_steps, lr):
    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))

print('get_lr function defined ✓')

### 4. DPO Training Loop

We set up a simplified DPO training loop:
1. Create a policy model and a frozen reference model (copy of policy).
2. For each batch, concatenate chosen and rejected inputs.
3. Forward through both models, compute log-probs, then the DPO loss.
4. Backpropagate only through the policy model.

In [ ]:
# Cell — DPO training setup

MAX_SEQ_LEN = 128
BATCH_SIZE = 4
NUM_STEPS = 30
LEARNING_RATE = 1e-4
BETA = 0.1

config = MiniMindConfig(
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=2,
    num_key_value_heads=2,
    head_dim=32,
    vocab_size=tokenizer.vocab_size,
)

# Policy model
torch.manual_seed(42)
policy_model = MiniMindForCausalLM(config).to(device)

# Reference model (frozen copy)
torch.manual_seed(42)
ref_model = MiniMindForCausalLM(config).to(device)
ref_model.eval()
ref_model.requires_grad_(False)

# Dataset and loader
dpo_dataset = DPODataset(tokenizer, max_length=MAX_SEQ_LEN, num_samples=200)
dpo_loader = DataLoader(dpo_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = optim.AdamW(policy_model.parameters(), lr=LEARNING_RATE)

total_params = sum(p.numel() for p in policy_model.parameters())
print(f'Policy model: {total_params:,} parameters')
print(f'Training for {NUM_STEPS} steps with batch_size={BATCH_SIZE}')

In [ ]:
# Cell — DPO training loop

policy_model.train()
losses = []
step_count = 0

for epoch in range(10):  # enough epochs to reach NUM_STEPS
    for batch in dpo_loader:
        if step_count >= NUM_STEPS:
            break

        x_chosen = batch['x_chosen'].to(device)
        x_rejected = batch['x_rejected'].to(device)
        y_chosen = batch['y_chosen'].to(device)
        y_rejected = batch['y_rejected'].to(device)
        mask_chosen = batch['mask_chosen'].to(device)
        mask_rejected = batch['mask_rejected'].to(device)

        x = torch.cat([x_chosen, x_rejected], dim=0)
        y = torch.cat([y_chosen, y_rejected], dim=0)
        mask = torch.cat([mask_chosen, mask_rejected], dim=0)

        step_count += 1
        lr = get_lr(step_count, NUM_STEPS, LEARNING_RATE)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # Reference forward (no grad)
        with torch.no_grad():
            ref_logits = ref_model(x).logits
        ref_lp = logits_to_log_probs(ref_logits, y)

        # Policy forward
        policy_logits = policy_model(x).logits
        policy_lp = logits_to_log_probs(policy_logits, y)

        loss = dpo_loss(ref_lp, policy_lp, mask.float(), beta=BETA)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item())
        if step_count % 10 == 0 or step_count == 1:
            print(f'Step {step_count:3d}/{NUM_STEPS} | Loss: {loss.item():.4f} | LR: {lr:.6f}')

    if step_count >= NUM_STEPS:
        break

print(f'\nTraining complete — {len(losses)} steps')

### ✅ Milestone 2: DPO Training — Loss Decreases

Over 30 training steps, the DPO loss should decrease, indicating the policy
model is learning to prefer chosen responses over rejected ones.

In [ ]:
# Cell — ✅ Milestone 2: DPO training — loss decreases

first_losses = losses[:5]
last_losses = losses[-5:]
first_avg = sum(first_losses) / len(first_losses)
last_avg = sum(last_losses) / len(last_losses)

print(f'First 5 losses avg: {first_avg:.4f}')
print(f'Last 5 losses avg:  {last_avg:.4f}')
print(f'Loss decrease:      {first_avg - last_avg:.4f}')

assert last_avg < first_avg, (
    f'DPO loss did not decrease! First avg: {first_avg:.4f}, Last avg: {last_avg:.4f}')
print(f'✅ Milestone 2 passed — DPO loss decreased from {first_avg:.4f} to {last_avg:.4f}')

## Part B — Group Relative Policy Optimization (35 min)

GRPO is a reinforcement learning approach for LLM alignment that:
1. Generates **multiple responses** per prompt (a *group*).
2. Scores them with a **reward function** (rule-based or model-based).
3. Normalizes rewards **within each group** to compute advantages.
4. Uses a **PPO-style clipped objective** to update the policy.

### Key equations

**Advantage normalization:**
$$A_i = \frac{r_i - \bar{r}_{\text{group}}}{\sigma_{\text{group}} + \epsilon}$$

**Clipped policy loss:**
$$\mathcal{L} = -\min\left(r_t \cdot A, \text{clip}(r_t, 1{\pm}\epsilon) \cdot A\right) + \beta \cdot \text{KL}$$

where $r_t = \exp(\log \pi_\theta - \log \pi_{\text{old}})$ and KL uses the Schulman approximation.

### 5. Reward Function

GRPO needs a reward signal for each generated response. In our simplified version,
we use **rule-based rewards** that check:
- Response length: appropriate length gets +0.5, too short/long gets -0.5
- Chain-of-thought structure: if `</think>` tag is present, the thinking section
  should be between 20–300 characters for a +1.0 bonus

In [ ]:
# Cell — calculate_rewards implementation

def calculate_rewards(prompts, responses):
    rewards = torch.zeros(len(responses))
    for i, response in enumerate(responses):
        rewards[i] += 0.5 if 20 <= len(response.strip()) <= 800 else -0.5
        if '</think>' in response:
            thinking, answer = response.split('</think>', 1)
            rewards[i] += 1.0 if 20 <= len(thinking.strip()) <= 300 else -0.5
    return rewards

print('calculate_rewards defined ✓')

### 6. GRPO Advantage Normalization & Clipped Policy Loss

The final piece combines:
1. **Group advantage normalization** — subtract group mean, divide by group std.
2. **Clipped ratio** — PPO-style clipping prevents too-large policy updates.
3. **KL penalty** — Schulman's approximation `exp(kl) - kl - 1` keeps the policy
   close to the reference model.
4. **Masked mean** — only completion tokens (up to EOS) contribute to the loss.

In [ ]:
# Cell — GRPO advantage normalization and clipped policy loss

# Simulate GRPO data
torch.manual_seed(42)
B = 4                  # number of prompts
num_generations = 3    # responses per prompt
T = 20                 # completion length
epsilon = 0.2
beta = 0.1

# Simulated reward scores
test_prompts = [f'prompt_{i}' for i in range(B)]
test_responses = []
for i in range(B):
    for j in range(num_generations):
        # Vary lengths and include some with <think> tags
        if j == 0:
            test_responses.append('A ' * 50)  # ~100 chars, good length
        elif j == 1:
            test_responses.append('Short')  # too short
        else:
            test_responses.append('Let me think step by step about this problem carefully. ' * 3 +
                                  '</think>The answer is 42.')  # has thinking

rewards = calculate_rewards(test_prompts, test_responses)
print(f'Rewards: {rewards}')
print(f'Rewards shape: {rewards.shape}')

# Simulated per-token log probs
per_token_logps = torch.randn(B * num_generations, T) * 0.1 - 2.0
old_per_token_logps = per_token_logps.detach() + torch.randn_like(per_token_logps) * 0.01
ref_per_token_logps = per_token_logps.detach() + torch.randn_like(per_token_logps) * 0.05
completion_mask = torch.ones(B * num_generations, T)
# Simulate some sequences ending early
completion_mask[1, 10:] = 0
completion_mask[4, 15:] = 0

# Advantage normalization
grouped_rewards = rewards.view(-1, num_generations)
mean_r = grouped_rewards.mean(dim=1).repeat_interleave(num_generations)
std_r = grouped_rewards.std(dim=1).repeat_interleave(num_generations)
advantages = (rewards - mean_r) / (std_r + 1e-4)

# Clipped policy gradient loss
ratio = torch.exp(per_token_logps - old_per_token_logps)
clipped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
per_token_loss1 = ratio * advantages.unsqueeze(1)
per_token_loss2 = clipped_ratio * advantages.unsqueeze(1)
kl_div = ref_per_token_logps - per_token_logps
per_token_kl = torch.exp(kl_div) - kl_div - 1
per_token_loss = -(torch.min(per_token_loss1, per_token_loss2) - beta * per_token_kl)
loss = ((per_token_loss * completion_mask).sum(dim=1) / completion_mask.sum(dim=1)).mean()

print(f'\nAdvantages: {advantages}')
print(f'Advantages mean: {advantages.mean().item():.4f}')
print(f'Loss: {loss.item():.4f}')

### ✅ Milestone 3: GRPO Advantages — Mean ≈ 0

After group normalization, advantages should have mean close to zero
(within each group, rewards are centered). We verify |mean| < 0.1.

In [ ]:
# Cell — ✅ Milestone 3: GRPO advantages — mean ≈ 0

adv_mean = advantages.mean().item()
print(f'Advantages mean:  {adv_mean:.4f}')
print(f'Advantages std:   {advantages.std().item():.4f}')
print(f'|mean| < 0.1?     {abs(adv_mean) < 0.1}')

assert abs(adv_mean) < 0.1, (
    f'Advantages mean should be close to 0, got {adv_mean:.4f}')
assert not torch.isnan(loss), f'Loss is NaN!'
print(f'✅ Milestone 3 passed — GRPO advantages mean ≈ 0 (got {adv_mean:.4f}), loss = {loss.item():.4f}')

## Wrap-up (5 min)

### What we built today

| Component | Key idea |
|-----------|----------|
| **logits_to_log_probs** | Extract per-token log-probs via log_softmax + gather |
| **dpo_loss** | Preference optimization without a reward model |
| **DPODataset.generate_loss_mask** | Mask non-assistant tokens for preference training |
| **calculate_rewards** | Rule-based reward shaping for RL alignment |
| **GRPO loss** | Group-normalized advantages + PPO-style clipped objective |

### Training pipeline so far

```
Pretrain → SFT → LoRA → DPO / GRPO
                         ↑ Session 4
```

### Key takeaways
- **DPO** is simpler (no RL loop) but needs curated preference pairs.
- **GRPO** is more flexible — any reward signal works — but requires generation during training.
- Both keep the policy close to a reference model to prevent catastrophic drift.